# How Many Measurements Does a Country Need Before Its IQB Score Means Anything?

This is the primary sample-size analysis. It defines one planning criterion, derives the
per-country requirement, and records the assumptions that limit the result. Read this notebook
before the companion volatility and review notebooks.

## The question

A country's IQB score is computed from however many M-Lab measurements that country produced in
the period — 36 million for India, 1 for Pitcairn. At some point a score stops being a measurement
and starts being a guess. Where is that point?

## The answer

**There is no single number, but there is a simple formula.** IQB is ten pass/fail threshold
tests, so a score can only be wrong if one of those tests lands on the wrong side. Whether it does
comes down to one comparison, in units of measurements:

$$\underbrace{\bigl|\,\pi n - (1-q)n\,\bigr|}_{\textbf{gap}}
\quad\text{versus}\quad
\underbrace{\sqrt{n\,\pi(1-\pi)}}_{\textbf{spread}}$$

where $\pi$ is the fraction of a country's measurements that pass a threshold and $q$ is the
percentile the score is reported at. Demanding the gap beat the spread with 95% confidence gives

$$n \;\gtrsim\; \frac{\pi(1-\pi)\,z^2}{\delta^2}, \qquad \delta = |\pi - (1-q)|$$

At a ±0.05 tolerance and 95% confidence, for the median country:

| what you publish | median country | 90th pct | covered at n≥10,000 | sufficient today |
|---|---|---|---|---|
| p50 alone | ~560 | ~17,000 | 87% | 185 / 245 |
| p25 / p50 / p75 | ~2,400 | ~86,000 | 72% | 138 / 245 |
| all nine levels | ~5,400 | ~114,000 | 61% | 111 / 245 |

Counts are **download-side measurements** — seven of the ten tests, both loss tests among them,
are measured there; the upload tests are evaluated at each country's own upload/download ratio
(0.82 at the median). §4 explains why the two cannot be pooled into one number.

**What "true score" means here.** Every requirement is the sample size at which the reported score
matches the score of that country's *24-month pooled* distribution. That is the right target for
capacity planning and the wrong one for auditing a single month: a country's own pass rate moves
between months by about as much as the margin these sample sizes are sized to resolve. §10a
recomputes everything month by month, and the requirement rises — median 706, P90 29,460. Read the
pooled numbers as a minimum.

## Four things that fall out of it

1. **The requirement belongs to a (country, percentile level) pair, not a country.** All nine
   levels are computed from the same measurements, but each asks a different question of them, so
   Kenya needs 182 measurements at p50 and 397,024 at p75. Gate per level and most countries keep
   most of their curve: the median country can support 8 of the 9 levels today.
2. **Some countries cannot be fixed by any sample size.** A country sitting exactly on a threshold
   is unpredictable no matter how many measurements you collect. That is a property of threshold scoring, not of data collection, and
   the answer is to publish a band.
3. **Under-measurement does not just add noise, it subtracts score.** A country near 1.0 can only
   be pulled down by a flip, so its reported score is biased low — and because every under-sampled
   country is pushed the same way at once, the bias survives ranking and averaging. 24 of 245
   countries currently carry a bias larger than the ±0.05 tolerance itself, the worst by −0.33.
   See §8b.
4. **The binding constraint is sometimes the summary, not the sample.** $\pi$ has to be
   reconstructed by interpolating between nine percentile knots, and for 22 of 245 countries that
   interpolation is less certain than the margin it is trying to measure. Ten `COUNTIF` columns in
   the aggregation would make this whole analysis exact — see §12.

## How to read this notebook

§3–5 build the answer in units of measurements, which is all most readers need. §6 makes it exact
and checks it against the production `IQBCalculator`. §7–9 apply it, and §8b covers the bias. §10–11
are about how far to trust it. §12 is what to do.


## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq
from scipy import stats

from iqb import IQBCalculator
from iqb.calculator.config import IQB_CONFIG

calculator = IQBCalculator()

DATA_DIR = Path("./data/v1")
OUT_DIR = Path("./analysis_outputs")
OUT_DIR.mkdir(exist_ok=True)

PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
QLEV = np.array(PERCENTILES) / 100.0
METRICS = ["download", "upload", "latency", "loss"]

# Which sample count actually backs each metric. Latency and loss percentiles are reported in
# the *downloads* table (the uploads table carries only upload_*), so three of the four metrics
# rest on the download-side count and only upload rests on the upload-side one.
BACKED_BY = {"download": "down", "upload": "up", "latency": "down", "loss": "down"}

# What "reliable" is going to mean: the level reported, how wrong is too wrong, how sure.
Q_HEADLINE = 0.50      # p50
TOL        = 0.05      # tolerated absolute error on the 0-1 IQB scale
CONF       = 0.95
TINY       = 1e-9

RAW_KEY = {"download": "download_throughput_mbps", "upload": "upload_throughput_mbps",
           "latency": "latency_ms", "loss": "packet_loss"}
HIGHER_BETTER = {"download": True, "upload": True, "latency": False, "loss": False}
UNIT = {"download": "Mbps", "upload": "Mbps", "latency": "ms", "loss": "frac"}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "grid.linestyle": "--"})
print(f"Reporting level p{int(Q_HEADLINE*100)} | tolerance +/-{TOL} | confidence {CONF:.0%}")

## 2. The Data

24 months of M-Lab measurements, pre-aggregated to one row per country per month. Download and
upload live in separate tables and are merged on `country_code`, taking the **minimum** of the two
sample counts — a country is only as well measured as its worse-sampled direction.

In [ ]:
def discover_periods(cache_root: Path):
    out = []
    for period_dir in sorted(cache_root.iterdir()):
        if not period_dir.is_dir():
            continue
        for end_dir in sorted(period_dir.iterdir()):
            dl = end_dir / "downloads_by_country" / "data.parquet"
            ul = end_dir / "uploads_by_country" / "data.parquet"
            if dl.exists() and ul.exists():
                out.append((f"{period_dir.name[:4]}-{period_dir.name[4:6]}", dl, ul))
    return out


def load_all(cache_root: Path = DATA_DIR) -> pd.DataFrame:
    frames = []
    for month, dl_path, ul_path in discover_periods(cache_root):
        if dl_path.stat().st_size < 1000 or ul_path.stat().st_size < 1000:
            continue
        dl = pq.read_table(dl_path).to_pandas()
        ul = pq.read_table(ul_path).to_pandas()
        m = dl.merge(ul, on="country_code", suffixes=("_down", "_up"))
        # Seven of the ten tests (both loss tests among them) are measured on the download side,
        # so that is the headline n; the upload tests carry their own count. Taking min() of the
        # two would understate the n behind latency and loss by ~23% at the median.
        m["n_down"] = m["sample_count_down"].astype(float)
        m["n_up"] = m["sample_count_up"].astype(float)
        m["sample_count"] = m["n_down"]
        m["up_ratio"] = m["n_up"] / m["n_down"].clip(lower=1)
        m["month"] = month
        frames.append(m)
    return pd.concat(frames, ignore_index=True)


def country_name(cc: str) -> str:
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc


df = load_all()
latest = df[df.month == df.month.max()].reset_index(drop=True)
cc_arr = df.country_code.to_numpy()
n_arr = df.sample_count.to_numpy(float)
nup_arr = df.n_up.to_numpy(float)
ratio_arr = df.up_ratio.to_numpy(float)

print(f"{len(df):,} country-months  ({df.country_code.nunique()} countries x "
      f"{df.month.nunique()} months, {df.month.min()} .. {df.month.max()})")
print(f"download-side measurements per country-month: median {df.n_down.median():,.0f}  |  "
      f"P10 {df.n_down.quantile(.1):,.0f}  |  P90 {df.n_down.quantile(.9):,.0f}")
print(f"upload-side is {df.up_ratio.median():.2f}x that at the median "
      f"(P10 {df.up_ratio.quantile(.1):.2f}x, P90 {df.up_ratio.quantile(.9):.2f}x) - so the two "
      f"directions\nare not interchangeable and every requirement below is quoted in "
      f"download-side measurements.")
print(f"\nThe spread is the core difficulty: the best-measured country has "
      f"{df.sample_count.max() / max(df.sample_count.min(), 1):,.0f}x the data of the worst.")

## 3. IQB Is Ten Yes/No Questions

`iqb.calculator` computes a weighted average of use-case scores, each a weighted average of
network-requirement scores, each one a **binary** pass/fail against a fixed threshold. Flatten the
three layers of weights and the whole configuration collapses to ten *(metric, threshold)* tests —
24 use-case × requirement combinations, but only 10 distinct tests, because use cases share
thresholds:

$$\mathrm{IQB} \;=\; \sum_{k=1}^{10} w_k \cdot
\mathbf{1}\{\text{metric}_k \text{ passes threshold}_k\}, \qquad \sum_k w_k = 1$$

This is what makes the question answerable. **The score cannot move a little, it can only jump.** An
estimated IQB differs from the true one by exactly the total weight of the tests that came out
wrong, so "how many measurements do I need?" becomes "how many do I need to get each test on the
right side of its threshold?"

It also explains the volatility seen in notebook 02: the lightest test accounts for 0.014 of the scale
and the heaviest 0.202, so a single flip of `loss < 0.01` moves a country's score by a fifth.

In [ ]:
def derive_atoms(cfg=IQB_CONFIG):
    # Flatten the nested config into {metric: [(threshold, weight), ...]}, easiest test first.
    ucw_total = sum(uc["w"] for uc in cfg["use cases"].values())
    agg = {}
    for uc in cfg["use cases"].values():
        nrw_total = sum(nr["w"] for nr in uc["network requirements"].values())
        for nr_name, nr in uc["network requirements"].items():
            short = next(k for k, v in RAW_KEY.items() if v == nr_name)
            key = (short, nr["threshold min"])
            agg[key] = agg.get(key, 0.0) + (uc["w"] / ucw_total) * (nr["w"] / nrw_total)
    out = {m: [] for m in METRICS}
    for (m, thr), w in agg.items():
        out[m].append((thr, w))
    for m in out:
        out[m].sort(key=lambda t: t[0], reverse=not HIGHER_BETTER[m])
    return out


ATOMS = derive_atoms()
ATOM_LABELS = [f"{m} {'>' if HIGHER_BETTER[m] else '<'} {thr}"
               for m in METRICS for thr, _ in ATOMS[m]]
ATOM_W = np.array([w for m in METRICS for _, w in ATOMS[m]])
HEAVY = ATOM_W > TOL          # tests that break the tolerance on their own

print(f"{'test':24s}{'weight':>8s}   {'share of the scale':<20s}")
for lbl, w in zip(ATOM_LABELS, ATOM_W):
    print(f"{lbl:24s}{w:8.4f}   {'#' * int(round(w * 100)):<20s}")
print(f"{'TOTAL':24s}{ATOM_W.sum():8.4f}")
print(f"\n{HEAVY.sum()} of the {len(ATOM_W)} tests are heavy enough that flipping one, alone, "
      f"breaks a +/-{TOL} tolerance.")

In [ ]:
# The flattening has to be exact, or nothing downstream is trustworthy.
def iqb_from_atoms(vals: dict) -> float:
    return sum(w * ((vals[m] > thr) if HIGHER_BETTER[m] else (vals[m] < thr))
               for m in METRICS for thr, w in ATOMS[m])


check = df.sample(400, random_state=0)
worst = max(
    abs(iqb_from_atoms({m: float(r[f"{m}_p{p}"]) for m in METRICS})
        - calculator.calculate_iqb_score(
            data={"m-lab": {RAW_KEY[m]: float(r[f"{m}_p{p}"]) for m in METRICS}}))
    for _, r in check.iterrows() for p in PERCENTILES)

print(f"max |flattened - IQBCalculator| over {len(check)} country-months x 9 levels: {worst:.2e}")

## 4. One Set of Measurements, Nine Questions

The parquet reports one `sample_count` per country-month per direction. Every one of the nine
percentiles of a metric — `upload_p1` through `upload_p99` — is computed from those same
measurements. There is no sense in which p99 is built from fewer observations than p50.

Two counts, not one: the downloads table carries `download_*`, `latency_*` and `loss_*`, the
uploads table only `upload_*`. So seven of the ten tests — both loss tests among them — rest on
the download-side count and three on the upload-side one. Requirements below are quoted in
**download-side measurements**, with the upload tests evaluated at that country's own
upload/download ratio.

What changes with the percentile level is **the question being asked of them.**

Evaluating IQB at p50 asks *does the median user pass this threshold?* Rewrite that and it says
exactly the same thing as *do more than 50% of measurements pass?* At p75 the question becomes
*do more than 25% pass?*, and at p99, *do more than 1% pass?*

| evaluating at | is really asking | hard when the pass rate is near |
|---|---|---|
| p25 | do more than 75% pass? | 75% |
| p50 | do more than 50% pass? | 50% |
| p75 | do more than 25% pass? | 25% |
| p99 | do more than 1% pass? | 1% |

So each level has its own danger zone, and a country is hard to score at that level only if one of
its ten pass rates happens to land in it. This is why the percentile level matters so much, and
why it is not a detail to be swept into a robustness appendix.

**Where the pass rate comes from.** The one input needed is $\pi$, the fraction of a country's
measurements that pass a given threshold. The parquet does not report it, so it has to be
recovered from the percentile knots by finding where the threshold falls between them. The cell
below does that arithmetic in the open for a single country and threshold; §6 generalises it, and
§10b shows that this reconstruction is the weakest link in the whole analysis.

In [ ]:
# Find a well-measured country where exactly one level is ambiguous and the rest are clear.
# Restricted to throughput thresholds so the interpolation arithmetic below stays simple and
# visible; section 6 generalises it to all four metrics.
cands = []
for i, row in latest.iterrows():
    for m in ("download", "upload"):
        n = float(row.n_up if BACKED_BY[m] == "up" else row.n_down)
        if not (10_000 <= n <= 200_000):
            continue
        for thr, w in ATOMS[m]:
            if w <= TOL:
                continue
            knots = np.array([row[f"{m}_p{p}"] for p in PERCENTILES], float)
            if not (knots[0] < thr < knots[-1]):
                continue
            j = int(np.searchsorted(knots, thr))
            frac = ((np.log(thr) - np.log(knots[j - 1]))
                    / (np.log(knots[j]) - np.log(knots[j - 1])))
            pi = 1 - (QLEV[j - 1] + frac * (QLEV[j] - QLEV[j - 1]))
            z = np.abs(pi * n - (1 - QLEV) * n) / np.sqrt(n * pi * (1 - pi))
            # exactly one level ambiguous, every other level far clear, and prefer a case where
            # the fragile level is NOT the headline - that is the point of this section
            if (z < 1).sum() == 1 and PERCENTILES[int(z.argmin())] != 50:
                cands.append((np.sort(z)[1], row.country_code, m, thr, pi, n, j))

sep, EX_CC, EX_METRIC, EX_THRESH, pi_ex, n_ex, j = max(cands)
row = latest[latest.country_code == EX_CC].iloc[0]
knots = np.array([row[f"{EX_METRIC}_p{p}"] for p in PERCENTILES], float)

print(f"{country_name(EX_CC)} ({EX_CC}), {latest.month.iloc[0]}: "
      f"{EX_METRIC} measurements = {int(n_ex):,}")
print("that ONE number is the n behind every percentile below\n")
for jj, (p, v) in enumerate(zip(PERCENTILES, knots)):
    flag = (f"   <-- the {EX_THRESH:g} {UNIT[EX_METRIC]} threshold falls in this gap"
            if jj == j else "")
    print(f"   {EX_METRIC}_p{p:<2d} = {v:>8.2f} {UNIT[EX_METRIC]}{flag}")

frac = ((np.log(EX_THRESH) - np.log(knots[j - 1])) / (np.log(knots[j]) - np.log(knots[j - 1])))
print(f"\ninterpolating between p{PERCENTILES[j-1]} and p{PERCENTILES[j]} on a log scale "
      f"(throughput spans decades):")
print(f"   log({EX_THRESH:g}) sits {frac:.3f} of the way from log({knots[j-1]:.2f}) "
      f"to log({knots[j]:.2f})")
print(f"   -> the threshold sits at quality level {1 - pi_ex:.3f}")
print(f"   -> pass rate pi = {pi_ex:.3f}")
print(f"   -> about {pi_ex * n_ex:,.0f} of its {int(n_ex):,} measurements beat "
      f"{EX_THRESH:g} {UNIT[EX_METRIC]}")

### 4b. The Same Measurements, Judged Against Nine Different Cut-Offs

Now put that pass count next to what each level actually demands. The level converts to a cut-off
in raw measurements — p50 asks whether the pass count clears $0.50n$, p75 whether it clears
$0.25n$ — and the score is right or wrong according to which side of that cut-off the count falls.

The pass count is itself a random draw. Re-run the month with a different set of users and it
would land somewhere else. How far it moves is the **spread**, developed in §5; for now it is just
a number of measurements, and the only thing that matters is whether the gap to the cut-off is
bigger than it.

Watch what happens across the nine rows below. The measurements, the threshold and the spread are
identical in every one — only the cut-off moves. Eight of the nine questions have an answer that is
not in doubt, and one of them is a 50/50 result, in a country with tens of thousands of measurements.
Which level is affected is an accident of where that country's pass rate happens to sit.

In [ ]:
passing = pi_ex * n_ex
spr = np.sqrt(n_ex * pi_ex * (1 - pi_ex))          # see §5

print(f"{country_name(EX_CC)}: {int(n_ex):,} measurements, of which {passing:,.0f} beat "
      f"{EX_THRESH:g} {UNIT[EX_METRIC]} {EX_METRIC}.")
print(f"Re-run the month and that count would move by about +/-{spr:,.0f}.\n")
print(f"{'level':>7s}{'is really asking':>26s}{'cut-off':>10s}{'gap':>10s}"
      f"{'gap in SDs':>16s}{'verdict':>12s}")
for lvl in PERCENTILES:
    cut = (1 - lvl / 100) * n_ex
    gap = passing - cut
    z = abs(gap) / spr
    verdict = "certain" if z > 3 else ("50/50" if z < 1 else "unclear")
    print(f"{'p' + str(lvl):>7s}{f'do >{100-lvl}% pass?':>26s}{cut:>10,.0f}{gap:>+10,.0f}"
          f"{z:>16.1f}{verdict:>12s}")

amb = [p for p in PERCENTILES
       if abs(passing - (1 - p / 100) * n_ex) / spr < 1]
print(f"\nSame measurements, same threshold, same spread of +/-{spr:,.0f}. Only the cut-off moves.")
print(f"At p{amb[0]} it falls almost exactly on the pass count, so that level is a 50/50 result even "
      f"with\n{int(n_ex):,} measurements in hand - while every other level is certain.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), constrained_layout=True)
amb_lvl = [p for p in PERCENTILES if abs(passing - (1 - p / 100) * n_ex) / spr < 1][0]

ax = axes[0]
tr = ax.get_xaxis_transform()          # x in data units, y in axes fraction
for lvl in PERCENTILES:
    cut = (1 - lvl / 100) * n_ex
    hot = lvl == amb_lvl
    ax.axvline(cut, color="#c53030" if hot else "#4a5568", lw=1.6 if hot else 1)
    ax.text(cut, 0.88, f"p{lvl}", ha="center", fontsize=8, transform=tr,
            color="#c53030" if hot else "#4a5568",
            fontweight="bold" if hot else "normal")
ax.axvline(passing, color="#c53030", lw=2.4)
ax.text(passing, 0.30, f" {passing:,.0f} measurements\n pass the threshold", fontsize=8.5,
        transform=tr, color="#c53030", va="center")
ax.set_xlim(-n_ex * 0.03, n_ex * 1.03)
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("number of measurements")
ax.set_title(f"Each percentile level is a different cut-off on the same count\n"
             f"{country_name(EX_CC)}, {EX_METRIC} > {EX_THRESH:g} {UNIT[EX_METRIC]}, "
             f"n = {int(n_ex):,}", fontsize=10)

zs = {lvl: ((1 - lvl / 100) * n_ex - passing) / spr for lvl in PERCENTILES}
next_nearest = min(abs(z) for lvl, z in zs.items() if lvl != amb_lvl)

ax = axes[1]
tr = ax.get_xaxis_transform()
ax.axvspan(-1.96, 1.96, color="#c53030", alpha=0.18)
ax.text(0, 0.10, "where the count\ncould plausibly land", ha="center", fontsize=8.5,
        transform=tr, color="#c53030")
ax.axvline(0, color="#c53030", lw=2.4)
for lvl, z in zs.items():
    if abs(z) < 11:
        ax.axvline(z, color="#c53030", lw=1.6)
        ax.text(z, 0.88, f"p{lvl}", ha="center", fontsize=9, transform=tr,
                color="#c53030", fontweight="bold")
ax.annotate(f"every other level is\n{next_nearest:.0f}+ standard deviations away, off this chart",
            xy=(0.97, 0.55), xycoords="axes fraction", ha="right", fontsize=8.5,
            color="#4a5568")
ax.set_xlim(-11, 11)
ax.set_yticks([])
ax.grid(False)
ax.set_xlabel("distance from the pass count, in standard deviations")
ax.set_title(f"Zoomed in: only p{amb_lvl}'s cut-off falls inside\nthe range the count could land in",
             fontsize=10)
plt.show()

print(f"Eight of the nine questions have an answer that is not in doubt, and one is a 50/50 result.")
print(f"Section 5 turns that picture into the sample size p{amb_lvl} would actually need.")

## 5. The Spread, and How It Turns Into a Sample Size

Each measurement either passes a threshold or does not, independently, with probability $\pi$. So
the number that pass out of $n$ is $\mathrm{Binomial}(n, \pi)$, and its standard deviation — the
spread — is

$$\text{spread} \;=\; \sqrt{n\,\pi(1-\pi)}$$

The level's cut-off is at $(1-q)n$ measurements, so the gap between the pass count and the cut-off
is $|\pi n - (1-q)n| = \delta n$, writing $\delta = |\pi - (1-q)|$ for the **margin** on the
pass-rate scale.

Everything follows from comparing the two. The gap grows like $n$ while the spread grows like
$\sqrt{n}$, so more measurements always help — but if the gap starts small, it takes a great many
of them to separate. Demanding the gap beat the spread by $z$ standard deviations:

$$\delta n \;>\; z\sqrt{n\,\pi(1-\pi)}
\qquad\Longleftrightarrow\qquad
n \;>\; \frac{\pi(1-\pi)\,z^2}{\delta^2}$$

Which is the ordinary sample size for a proportion. Note what does **not** appear: the metric's
units, its variance, how heavy-tailed it is. A country with wildly dispersed download speeds is no
harder to score than a stable one — only the margin matters.

In [ ]:
def spread(n, pi):
    """Standard deviation of the number of measurements passing a threshold."""
    return np.sqrt(n * pi * (1 - pi))


def certainty(n, pi, level):
    """How many standard deviations separate the pass count from the cut-off this level asks about."""
    return abs(pi * n - (1 - level) * n) / spread(n, pi)


def samples_needed(pi, level, conf=CONF):
    """Smallest n at which that comparison is certain with confidence `conf`."""
    z = stats.norm.ppf(1 - (1 - conf) / 2)
    delta = abs(pi - (1 - level))
    return np.inf if delta == 0 else int(np.ceil(pi * (1 - pi) * z ** 2 / delta ** 2))


print(f"{country_name(EX_CC)}, {EX_METRIC} > {EX_THRESH:g} {UNIT[EX_METRIC]}, "
      f"pass rate {pi_ex:.3f}  (it has {int(n_ex):,})")
print(f"{'level':>7s}{'margin':>10s}{'n needed':>12s}{'SDs at that n':>20s}")
for lvl in [0.25, 0.50, 0.75, 0.90, 0.99]:
    need = samples_needed(pi_ex, lvl)
    z_at = certainty(need, pi_ex, lvl) if np.isfinite(need) else np.nan
    print(f"{'p' + str(int(lvl * 100)):>7s}{abs(pi_ex - (1 - lvl)):>10.4f}"
          f"{need:>12,d}{z_at:>20.2f}")
print("\nThe last column is the check: at the required n the gap is 1.96 standard deviations, by construction.")
need_amb = samples_needed(pi_ex, amb_lvl / 100)
print(f"\nSo the level that came out a 50/50 result in section 4, p{amb_lvl}, would need "
      f"{need_amb:,} measurements\nto become certain - {need_amb / n_ex:.0f}x what {country_name(EX_CC)} "
      f"has - while every other level is already secure\non a few dozen.")
print(f"\nA feel for the scale — margin to samples needed, at {CONF:.0%}:")
for d in [0.30, 0.10, 0.03, 0.01, 0.003, 0.001]:
    print(f"   margin {d:<6.3f} -> {0.25 * (1.96 / d) ** 2:>12,.0f}")
print("Every factor of 10 in measurements improves margin resolution by only ~3x.")

## 6. From One Threshold to the Whole Score

§4–5 handled one threshold in one country. A score has ten, so three things need generalising:
recovering $\pi$ for every test, combining ten tests into a score, and replacing the normal
approximation with something exact at small $n$.

### 6a. Recovering every pass rate

Two details make this uniform across metrics.

**The percentile index is a quality rank.** It ranks users within a country by connection quality:
p25 is the 25th-percentile user there, p99 among the best served. For download that is the
ordinary 25th percentile of speed; for latency and loss the pipeline has already flipped the
direction, so `latency_p1` is the *worst* latency in the country and `latency_p99` the best. That
makes one rule cover all four metrics — the $q$-quantile passes test $k$ exactly when
$q > 1 - \pi_k$ — and it is checked below against every country-month rather than assumed.

**A threshold outside the reported range gives only a bound.** If a threshold is better than a
country's p99, all we learn is that at most 1% of measurements pass, not how much less. Such a
value must be pushed past the edge rather than set exactly at it: returning exactly 0.01 would place
the cut point precisely at the p99 reporting level and create a permanently unpredictable test there.

In [ ]:
def to_quality(metric, v):
    # Map raw values into quality space, where higher is better and passing means y > t.
    v = np.asarray(v, dtype=float)
    if metric in ("download", "upload"):
        return np.log(np.maximum(v, TINY))       # log: throughput spans decades
    if metric == "latency":
        return -np.log(np.maximum(v, TINY))
    return -v                                     # loss: linear, it has a point mass at 0


for m in METRICS:
    Y = to_quality(m, df[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
    bad = (np.diff(Y, axis=1) < -1e-12).any(axis=1).sum()
    print(f"{m:9s}: country-months where quality decreases with the percentile index: "
          f"{bad} / {len(df)}")


def pass_prob(y_knots, t):
    # P(one measurement passes) = 1 - F(t), read off the 9 knots.
    #
    # The knots define a piecewise-linear CDF. Repeated values (loss = 0 across p75..p99) are a
    # point mass, so the CDF jumps there. Between adjacent distinct knots the definition of a
    # quantile brackets F(t) in [q_max(left), q_min(right)) - see §10b. Outside the knot range
    # only a bound is available, pushed past the edge rather than set exactly at it.
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    q_lo = np.array([QLEV[y == v].min() for v in uy])
    q_hi = np.array([QLEV[y == v].max() for v in uy])
    if t <= uy[0]:
        return 0.995                                   # F <= 0.01 -> pi >= 0.99 (bound only)
    if t >= uy[-1]:
        return 0.005                                   # F >= 0.99 -> pi <= 0.01 (bound only)
    i = np.searchsorted(uy, t) - 1
    F = q_hi[i] + (q_lo[i + 1] - q_hi[i]) * (t - uy[i]) / (uy[i + 1] - uy[i])
    return float(np.clip(1.0 - F, 0.005, 0.995))


def pass_prob_matrix(frame) -> np.ndarray:
    cols = []
    for m in METRICS:
        Y = to_quality(m, frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr))
            cols.append(np.array([pass_prob(Y[i], t) for i in range(len(Y))]))
    return np.column_stack(cols)


def threshold_in_range(frame) -> np.ndarray:
    out = []
    for m in METRICS:
        Y = to_quality(m, frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float))
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr))
            out.append((Y.min(axis=1) < t) & (t < Y.max(axis=1)))
    return np.column_stack(out)


P_all = pass_prob_matrix(df)
in_range = threshold_in_range(df)
print(f"\npass-rate matrix: {P_all.shape}")
print(f"thresholds outside the reported p1-p99 range (pass rate known only as a bound): "
      f"{(~in_range).mean():.1%} of country-month-threshold pairs")

### 6b. The Exact Sampling Distribution

The spread in §5 is a normal approximation to a binomial, which is fine at $n = 20{,}000$ and
poor at $n = 20$. The exact version is a standard order-statistic result: for $n$ i.i.d.
measurements the sample $q$-quantile is the $r$-th order statistic, $r = \lceil qn \rceil$, and its
position on the true CDF is Beta distributed,

$$U \;=\; F(\hat{Q}_q) \;\sim\; \mathrm{Beta}(r,\; n - r + 1)$$

so test $k$ looks like a pass exactly when $U > c_k$, writing $c_k = 1 - \pi_k$. This is the same
comparison as §4 — is the count above the cut-off — with the binomial handled exactly instead of
approximated.

Combining the ten tests is then easy. Within a metric the thresholds are nested (passing 25 Mbps
implies passing 10), so each metric contributes one of at most four values, and across four
metrics the estimate lives on a fixed lattice of $3 \times 4 \times 4 \times 3 = 144$ points.
Every probability on it is a difference of Beta survival functions, so the whole sampling
distribution is available in closed form for any $n$.

The metrics are treated as independent. They are certainly correlated in reality, but §10c shows
it makes no difference to the answer, because the requirement is set by a single binding test.

In [ ]:
BANDS = {m: np.concatenate([[0.0], np.cumsum([w for _, w in ATOMS[m]])]) for m in METRICS}
KS = [len(ATOMS[m]) for m in METRICS]
LATTICE = sum(g.ravel() for g in np.meshgrid(*[BANDS[m] for m in METRICS], indexing="ij"))
LAT_ORDER = np.argsort(LATTICE)


def sampling_dist(C, q, n, ratio=1.0):
    # [rows x 144] exact distribution of the IQB estimate. C = cut points (1 - pass rate).
    # n is the DOWNLOAD-side count; the upload tests are evaluated at n * ratio, because they are
    # measured on a different set of tests (see BACKED_BY).
    rows = len(C)
    n_dn = np.full(rows, max(int(n), 1))
    n_up = np.maximum(np.round(n_dn * np.broadcast_to(np.asarray(ratio, float), (rows,))), 1)
    parts, off = [], 0
    for m, K in zip(METRICS, KS):
        n_m = (n_up if BACKED_BY[m] == "up" else n_dn).astype(np.int64)
        r = np.clip(np.ceil(q * n_m), 1, n_m).astype(np.int64)
        # P(U > c_k): test k looks like a pass. Exact for discrete F too - loss has an atom at 0.
        s = stats.beta.sf(C[:, off:off + K], r[:, None], (n_m - r + 1)[:, None])
        off += K
        p = np.empty((rows, K + 1))
        p[:, 0] = 1.0 - s[:, 0]
        for j in range(1, K):
            p[:, j] = s[:, j - 1] - s[:, j]
        p[:, K] = s[:, K - 1]
        parts.append(np.clip(p, 0.0, None))
    J = parts[0]
    for p in parts[1:]:
        J = (J[:, :, None] * p[:, None, :]).reshape(rows, -1)
    return J


def true_scores(P, q):
    t = np.zeros(len(P))
    for k in range(P.shape[1]):
        t += ATOM_W[k] * (q > 1.0 - P[:, k])
    return t


def p_within(C, t, q, n, ratio=1.0, tol=TOL):
    J = sampling_dist(C, q, n, ratio)
    return (J * (np.abs(LATTICE[None, :] - t[:, None]) <= tol + 1e-12)).sum(axis=1)


def dist_stats(C, t, q, n, ratio=1.0, tol=TOL):
    # sd, mean absolute error, MAD, bias and P(|error| <= tol).
    J = sampling_dist(C, q, n, ratio)
    mean = (J * LATTICE).sum(axis=1)
    sd = np.sqrt((J * (LATTICE[None, :] - mean[:, None]) ** 2).sum(axis=1))
    mae = (J * np.abs(LATTICE[None, :] - t[:, None])).sum(axis=1)
    pok = (J * (np.abs(LATTICE[None, :] - t[:, None]) <= tol + 1e-12)).sum(axis=1)
    L, Js = LATTICE[LAT_ORDER], J[:, LAT_ORDER]
    cdf = np.cumsum(Js, axis=1) / Js.sum(axis=1, keepdims=True)
    med = L[np.argmax(cdf >= 0.5, axis=1)]
    d = np.abs(L[None, :] - med[:, None])
    order = np.argsort(d, axis=1)
    dc = np.take_along_axis(Js, order, axis=1).cumsum(axis=1)
    dc /= dc[:, -1:]
    mad = np.take_along_axis(d, order, axis=1)[np.arange(len(C)), np.argmax(dc >= 0.5, axis=1)]
    return dict(sd=sd, mae=mae, mad=mad, p_within=pok, bias=mean - t)


print(f"IQB estimates live on a {len(LATTICE)}-point lattice, "
      f"{LATTICE.min():.3f} .. {LATTICE.max():.3f}")
print(f"smallest gap between adjacent lattice points: "
      f"{np.diff(np.unique(LATTICE.round(6))).min():.4f}")

### 6c. Checking It Against the Production Calculator

Below, synthetic measurements are drawn from each country's reconstructed distribution, the sample
percentile is taken, and the four resulting metric values are scored with `IQBCalculator` — the
same code path production uses. Monte-Carlo and closed form should agree to within simulation
error (±0.01 at 2,000 reps).

**What this does and does not check.** Both sides draw from the *same* reconstructed CDF, so this
validates the Beta algebra, the 144-point lattice and the config flattening — not the
reconstruction itself, which §10b shows is the weakest link. No within-month replicate exists in a
monthly pre-aggregate, so the reconstruction cannot be validated from this data at all; that is a
second argument for §12's recommendation 2.

The countries are chosen to span the range, including the tightest-margin well-sampled country in
the latest month, which is the case most likely to expose an error.

In [ ]:
rng = np.random.default_rng(0)

ratio_lat = latest.up_ratio.to_numpy(float)
P_lat = pass_prob_matrix(latest)
margin_lat = np.where(HEAVY, np.abs(P_lat - (1 - Q_HEADLINE)), np.inf).min(axis=1)
well_sampled = np.where(latest.n_down.to_numpy() >= 1000)[0]
TIGHT_CC = latest.country_code.iloc[well_sampled[np.argmin(margin_lat[well_sampled])]]


def inv_quality_cdf(y_knots, u):
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    q_lo = np.array([QLEV[y == v].min() for v in uy])
    q_hi = np.array([QLEV[y == v].max() for v in uy])
    if u <= q_lo[0]:
        return uy[0]
    if u >= q_hi[-1]:
        return uy[-1]
    i = int(np.clip(np.searchsorted(q_hi, u, side="right") - 1, 0, len(uy) - 2))
    if u <= q_hi[i]:
        return uy[i]
    span = q_lo[i + 1] - q_hi[i]
    return uy[i] + (0.0 if span <= 0 else (u - q_hi[i]) / span) * (uy[i + 1] - uy[i])


def from_quality(metric, y):
    if metric in ("download", "upload"):
        return float(np.exp(y))
    if metric == "latency":
        return float(np.exp(-y))
    return float(-y)


def monte_carlo(row, q, n, ratio, reps=2000):
    n_m = {m: max(int(round(n * ratio)) if BACKED_BY[m] == "up" else int(n), 1) for m in METRICS}
    r_m = {m: int(np.clip(np.ceil(q * v), 1, v)) for m, v in n_m.items()}
    knots = {m: to_quality(m, [row[f"{m}_p{p}"] for p in PERCENTILES]) for m in METRICS}
    out = np.empty(reps)
    for i in range(reps):
        vals = {m: from_quality(m, inv_quality_cdf(
                    knots[m], np.sort(rng.random(n_m[m]))[r_m[m] - 1])) for m in METRICS}
        out[i] = calculator.calculate_iqb_score(
            data={"m-lab": {RAW_KEY[k]: v for k, v in vals.items()}})
    return out


print(f"tightest-margin well-sampled country this month: {TIGHT_CC} "
      f"({country_name(TIGHT_CC)}), margin {margin_lat[well_sampled].min():.4f}\n")
print(f"{'country':>8s} {'n':>6s} | {'true':>6s} {'MC mean':>8s} {'exact':>7s} | "
      f"{'MC sd':>6s} {'exact':>7s} | {'MC P(ok)':>9s} {'exact':>7s}")
for cc in ["US", "BR", "IN", "NG", TIGHT_CC]:
    sub = latest[latest.country_code == cc]
    if sub.empty:
        continue
    Psub = pass_prob_matrix(sub)
    rt = float(sub.up_ratio.iloc[0])
    C, t = 1 - Psub, true_scores(Psub, Q_HEADLINE)
    for n in (100, 1000):
        st = dist_stats(C, t, Q_HEADLINE, n, rt)
        exact_mean = float((sampling_dist(C, Q_HEADLINE, n, rt) * LATTICE).sum())
        sim = monte_carlo(sub.iloc[0], Q_HEADLINE, n, rt)
        print(f"{cc:>8s} {n:6d} | {t[0]:6.3f} {sim.mean():8.3f} {exact_mean:7.3f} | "
              f"{sim.std():6.3f} {st['sd'][0]:7.3f} | "
              f"{np.mean(np.abs(sim - t[0]) <= TOL + 1e-12):9.3f} {st['p_within'][0]:7.3f}")

## 7. What "Reliable" Should Mean

Two criteria, answering different questions.

**Median absolute deviation of the sampling distribution.** MAD is the natural robust width here
because the estimate is lattice-valued: most of the time it lands exactly on the true score, and
occasionally one test flips and it jumps by up to 0.2. A standard deviation is dominated by those
rare jumps; MAD ignores them. `MAD = 0` has a clean reading — *more than half the time you get
exactly the right score.*

That robustness is also its limit: MAD saturates at 0 and then stops distinguishing a country that
is right 51% of the time from one right 99.9% of the time. So MAD describes typical behaviour, and
a tail criterion sets the requirement:

$$\Pr\bigl(|\widehat{\mathrm{IQB}} - \mathrm{IQB}| \le \tau\bigr) \;\ge\; 0.95,
\qquad \tau = 0.05$$

$\tau = 0.05$ is strict on purpose — smaller than 6 of the 10 test weights, so in practice it
means "no consequential test flipped", though strictly it is a bound on the *net* error and so
also forgives two flips that cancel. §10c reports how the answer moves at 0.02 and 0.10.

Two features of the curves below are real rather than plotting artefacts. The **sawtooth at small
n** comes from the rank convention: the sample quantile is order statistic $r = \lceil qn \rceil$,
so $r/n$ jumps around $q$ and reliability can briefly worsen with one more measurement. And the
estimate is **biased, not merely noisy** — a country scoring near 1.0 can only be pulled down by a
flip, so at small $n$ its score is systematically understated, and that does not average out
across countries.

In [ ]:
NGRID_PLOT = np.unique(np.round(np.logspace(0.7, 5.5, 40)).astype(int))
picks = [well_sampled[np.argsort(margin_lat[well_sampled])[int(f * (len(well_sampled) - 1))]]
         for f in (0.90, 0.45, 0.02)]
labels = ["wide margin", "typical margin", "on the threshold"]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, (ax, title) in enumerate(zip(axes, ["MAD of the estimate", "mean absolute error",
                                           f"P(|error| <= {TOL})"])):
    for idx, lab in zip(picks, labels):
        C = 1 - P_lat[[idx]]
        t = true_scores(P_lat[[idx]], Q_HEADLINE)
        rt = float(ratio_lat[idx])
        ys = [[dist_stats(C, t, Q_HEADLINE, int(n), rt)[k][0] for n in NGRID_PLOT]
              for k in ["mad", "mae", "p_within"]][i]
        cc = latest.country_code.iloc[idx]
        ax.plot(NGRID_PLOT, ys, marker="o", ms=3,
                label=f"{cc} — {lab} (margin {margin_lat[idx]:.3f})")
    ax.set_xscale("log")
    ax.set_xlabel("measurements in the period")
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7)
axes[2].axhline(CONF, color="crimson", ls="--", lw=1)
axes[2].set_ylim(0, 1.02)
fig.suptitle(f"Same shape, shifted: margin sets the position, sample size moves you along it "
             f"(p{int(Q_HEADLINE*100)}, {latest.month.iloc[0]})", fontsize=11)
fig.tight_layout()
plt.show()

## 8. How Many Measurements Does Each Country Need?

### Estimating the margin without circularity

The margin has to come from the same percentile data whose reliability is in question, and a
country with three measurements gives a useless estimate of its own pass rates. Pooling months
fixes it, and does so exactly.

Notebook 02 establishes that monthly distributions must be combined by averaging **CDFs**, since
the mean of two p50s is not the p50 of the pooled data. A pass rate is that same CDF evaluated at a
**fixed** $x$ — the threshold — so

$$\pi_k^{\text{pooled}} \;=\; 1 - F_{\text{merged}}(t_k)
\;=\; \sum_i w_i\bigl(1 - F_i(t_k)\bigr) \;=\; \sum_i w_i \pi_{k,i}$$

is an identity, not an approximation. What cannot be averaged is a *quantile*, a position on the
vertical axis that moves when the distribution changes shape. A pass rate sits on the horizontal
axis at a pinned $x$, and those add linearly. This is notebook 02's rule applied where it is
exact, and it skips the inversion step, because nothing here needs percentiles back out. §10b
prices what that step costs if taken anyway.

Pooling 24 months gives each country an effective sample 24× larger for the purpose of locating
its thresholds. The cost is that a country whose performance genuinely changes is represented by its long-run mixture —
the right object for capacity planning, but it understates the requirement in a month when that
country sits closer to a threshold than usual. §10a measures how often that happens.

In [ ]:
NGRID = np.unique(np.round(np.logspace(0, 8.3, 170)).astype(int))


def required_n(C, t, q, ratio=1.0, tol=TOL, conf=CONF):
    # Smallest grid n (download-side) with P(|error| <= tol) >= conf there AND at every larger n.
    ok = np.array([p_within(C, t, q, int(n), ratio, tol) >= conf for n in NGRID])
    allok = np.empty_like(ok)
    acc = np.ones(len(C), bool)
    for i in range(len(NGRID) - 1, -1, -1):
        acc = acc & ok[i]
        allok[i] = acc
    return np.where(allok.any(axis=0), NGRID[np.argmax(allok, axis=0)], np.inf).astype(float)


codes, P_rows, n_med, n_months, ratio_rows = [], [], [], [], []
for c in pd.unique(cc_arr):
    m = cc_arr == c
    w = n_arr[m]
    w = w / w.sum() if w.sum() > 0 else np.full(m.sum(), 1 / m.sum())
    codes.append(c)
    P_rows.append((P_all[m] * w[:, None]).sum(axis=0))
    n_med.append(np.median(n_arr[m]))
    n_months.append(int(m.sum()))
    ratio_rows.append(nup_arr[m].sum() / max(n_arr[m].sum(), 1.0))

P_c = np.array(P_rows)
C_c = 1 - P_c
n_med = np.array(n_med)
ratio_c = np.array(ratio_rows)
t_c = true_scores(P_c, Q_HEADLINE)

margin_c = np.abs(P_c - (1 - Q_HEADLINE))
kbind = np.where(HEAVY, margin_c, np.inf).argmin(axis=1)      # reused by 10b
margin_heavy = np.where(HEAVY, margin_c, np.inf).min(axis=1)
binding = np.array(ATOM_LABELS)[kbind]

TOLS = [0.02, 0.05, 0.10]
req = pd.DataFrame({"cc": codes, "country": [country_name(c) for c in codes],
                    "n_month_median": n_med, "months": n_months, "iqb_p50": t_c,
                    "up_ratio": ratio_c, "margin": margin_heavy, "binding_test": binding})
for tol in TOLS:
    req[f"n_req_{tol}"] = required_n(C_c, t_c, Q_HEADLINE, ratio_c, tol)

sd, mae, mad, pok = (np.empty(len(req)) for _ in range(4))
for n in np.unique(np.maximum(n_med, 1).astype(int)):
    idx = np.where(np.maximum(n_med, 1).astype(int) == n)[0]
    s = dist_stats(C_c[idx], t_c[idx], Q_HEADLINE, int(n), ratio_c[idx])
    sd[idx], mae[idx], mad[idx], pok[idx] = s["sd"], s["mae"], s["mad"], s["p_within"]
req["sd_at_current_n"], req["mae_at_current_n"] = sd, mae
req["mad_at_current_n"], req["p_ok_at_current_n"] = mad, pok

print(f"=== download-side measurements needed per country, p{int(Q_HEADLINE*100)}, "
      f"{CONF:.0%} confidence ===")
print(f"{'':>12s}" + "".join(f"{'tol=' + str(x):>13s}" for x in TOLS))
for qq in [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]:
    vals = [req[f"n_req_{x}"].replace(np.inf, np.nan).quantile(qq) for x in TOLS]
    print(f"{'P' + str(int(qq * 100)):>12s}" + "".join(f"{v:>13,.0f}" for v in vals))
print(f"{'unreachable':>12s}" + "".join(f"{np.isinf(req[f'n_req_{x}']).sum():>13d}" for x in TOLS))
print("\nbinding test across countries (which threshold sets the requirement):")
print(req.binding_test.value_counts().to_string())

In [ ]:
print(f"=== coverage: share of the {len(req)} countries reliable at a given sample size ===")
print(f"{'n':>12s}" + "".join(f"{'tol=' + str(x):>12s}" for x in TOLS))
for n in [100, 300, 1000, 3000, 10_000, 30_000, 100_000, 1_000_000]:
    print(f"{n:>12,d}" + "".join(f"{(req[f'n_req_{x}'] <= n).mean():>12.1%}" for x in TOLS))

print(f"\n=== where the {len(req)} countries stand today (tol={TOL}, p50) ===")
for k in [1, 2, 3, 6, 12, 24]:
    ok = req.n_month_median * k >= req["n_req_0.05"]
    print(f"  pooling {k:2d} month(s): {ok.sum():3d} countries meet the bar ({ok.mean():5.1%})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

ax = axes[0]
for tol, col in zip(TOLS, ["#2b6cb0", "#dd6b20", "#38a169"]):
    v = np.sort(req[f"n_req_{tol}"].replace(np.inf, 1e9).to_numpy())
    ax.plot(v, np.arange(1, len(v) + 1) / len(v), lw=2, color=col, label=f"tolerance +/-{tol}")
ax.set_xscale("log")
ax.set_xlim(10, 1e7)
ax.set_xlabel("measurements per period required")
ax.set_ylabel("share of countries covered")
ax.set_title(f"What a sample-size cutoff gives you (p{int(Q_HEADLINE*100)}, {CONF:.0%})", fontsize=10)
for g in (1000, 10_000):
    ax.axvline(g, color="grey", ls=":", lw=1)
    ax.text(g, 0.02, f" n={g:,}", fontsize=8, color="grey")
ax.legend(fontsize=9, loc="lower right")

ax = axes[1]
fin = np.isfinite(req["n_req_0.05"])
short = req.n_month_median < req["n_req_0.05"]
ax.scatter(req.n_month_median[fin & ~short], req.loc[fin & ~short, "n_req_0.05"],
           s=16, alpha=0.6, color="#38a169", label="current n is enough")
ax.scatter(req.n_month_median[fin & short], req.loc[fin & short, "n_req_0.05"],
           s=16, alpha=0.6, color="#c53030", label="current n is short")
ax.plot([1, 1e8], [1, 1e8], color="black", lw=1, ls="--", label="required = available")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(1, 1e7)
ax.set_ylim(1, 1e8)
ax.set_xlabel("median monthly measurements")
ax.set_ylabel("measurements required")
ax.set_title("Which countries are actually short of data", fontsize=10)
ax.legend(fontsize=8, loc="upper left")
for _, r in req[fin & short].nlargest(6, "n_req_0.05").iterrows():
    ax.annotate(r.cc, (max(r.n_month_median, 1.2), r["n_req_0.05"]), fontsize=7)
fig.tight_layout()
plt.show()
print(f"countries short of data at monthly cadence (p50 only): {short.sum()} / {len(req)}")

### 8b. Falling Short Does Not Just Add Noise — It Subtracts Score

Everything so far has treated error as symmetric noise. It is not. A country scoring near 1.0 has
cannot score any higher: every test that can flip can only flip downward, so its expected score
sits *below* its true one. Near 0 the reverse holds. The estimate is pulled toward the middle of
the lattice — and since 0 is bad and 1 is good, **an under-measured country is published as worse
than it is.**

This deserves separate treatment from the variance for two reasons. Widening the tolerance does
not shrink it. And it does not average out across countries: every under-sampled country is pushed
the same direction at the same time, so league tables, regional averages and year-on-year
comparisons all inherit it.


In [ ]:
# Bias of the reported score - the part a wider tolerance does not fix.
print(f"{'n':>9s}{'mean bias':>12s}{'if true > 0.8':>15s}{'if true < 0.4':>15s}{'|bias| > tol':>14s}")
for n in (50, 100, 300, 1_000, 3_000, 10_000):
    J = sampling_dist(C_c, Q_HEADLINE, n, ratio_c)
    b = (J * LATTICE).sum(axis=1) - t_c
    print(f"{n:>9,d}{b.mean():>12.3f}{b[t_c > 0.8].mean():>15.3f}{b[t_c < 0.4].mean():>15.3f}"
          f"{(np.abs(b) > TOL).mean():>14.1%}")

bias_now = np.empty(len(codes))
for n in np.unique(np.maximum(n_med, 1).astype(int)):
    idx = np.where(np.maximum(n_med, 1).astype(int) == n)[0]
    J = sampling_dist(C_c[idx], Q_HEADLINE, int(n), ratio_c[idx])
    bias_now[idx] = (J * LATTICE).sum(axis=1) - t_c[idx]
req["bias_at_current_n"] = bias_now

bad = np.abs(bias_now) > TOL
print(f"\nAt each country's current monthly volume, {bad.sum()} of {len(codes)} countries carry a "
      f"bias larger than the\n+/-{TOL} tolerance itself - mean {bias_now[bad].mean():+.3f} among "
      f"them, worst {bias_now.min():+.3f}. Across all {len(codes)}\ncountries it averages "
      f"{bias_now.mean():+.4f}: it does not cancel.")
print(f"\n{'the six most understated countries today':>40s}")
print(f"{'country':>28s}{'n/month':>10s}{'true':>8s}{'reported':>10s}{'bias':>9s}")
for i in np.argsort(bias_now)[:6]:
    print(f"{req.country.iloc[i][:27]:>28s}{n_med[i]:>10,.0f}{t_c[i]:>8.3f}"
          f"{t_c[i] + bias_now[i]:>10.3f}{bias_now[i]:>+9.3f}")


## 9. It Depends Which Percentiles You Publish

Everything so far sets the requirement for one reporting level. The pipeline computes nine, and if
all nine are published then all nine have to be trustworthy — so the country's requirement is the
**maximum across the levels published**, not the p50 one.

The reason this matters more than the per-level medians suggest goes back to §4. Each level puts a
different pass rate at risk, and nine levels times ten thresholds is ninety chances for a country
to sit near a boundary. Most countries take one. So although p50 is the hardest level *on
average*, it is the binding level for well under half of countries *individually*.

In [ ]:
R_lvl = np.column_stack([required_n(C_c, true_scores(P_c, q), q, ratio_c) for q in QLEV])
R_fin = np.where(np.isinf(R_lvl), np.nan, R_lvl)
# An unreachable level must make the whole-curve requirement infinite rather than be dropped by nanmax:
# a country that cannot support p75 at any n does not "support all nine at 286 measurements".
curve_max = np.where(np.isinf(R_lvl).any(axis=1), np.inf, np.nanmax(R_fin, axis=1))
binding_level = np.array([f"p{PERCENTILES[i]}" for i in
                          np.where(np.isinf(R_lvl), np.inf,
                                   np.nan_to_num(R_fin, nan=-1.0)).argmax(axis=1)])
i50 = PERCENTILES.index(50)

print("=== requirement at each level the pipeline computes ===")
print(f"{'level':>7s}{'P25':>10s}{'P50':>10s}{'P75':>10s}{'P90':>10s}{'OK today':>18s}")
for j, p_ in enumerate(PERCENTILES):
    s = pd.Series(R_lvl[:, j]).replace(np.inf, np.nan)
    print(f"{'p' + str(p_):>7s}" + "".join(f"{s.quantile(x):>10,.0f}" for x in (.25, .5, .75, .9))
          + f"{(n_med >= R_lvl[:, j]).sum():>12d}/{len(n_med):<5d}")

def _scenario(levels):
    j = [PERCENTILES.index(x) for x in levels]
    return np.where(np.isinf(R_lvl[:, j]).any(axis=1), np.inf, np.nanmax(R_fin[:, j], axis=1))


SCENARIOS = {
    "p50 alone": R_lvl[:, i50],
    "p25 / p50 / p75": _scenario((25, 50, 75)),
    "all nine levels": curve_max,
}
print(f"\n=== the requirement depends on what gets published ===")
print(f"{'published':22s}" + "".join(f"{k:>10s}" for k in ["P50", "P90"])
      + f"{'cov@1k':>9s}{'cov@10k':>9s}{'sufficient today':>19s}")
for name, v in SCENARIOS.items():
    s = pd.Series(v).replace(np.inf, np.nan)
    filled = np.nan_to_num(v, nan=np.inf)
    print(f"{name:22s}" + "".join(f"{s.quantile(x):>10,.0f}" for x in (.5, .9))
          + f"{(filled <= 1000).mean():>9.1%}{(filled <= 10_000).mean():>9.1%}"
          + f"{(n_med >= filled).sum():>13d} / {len(n_med):<5d}")
print(f"\ncountries with at least one level unreachable at any n: "
      f"{int(np.isinf(R_lvl).any(axis=1).sum())} (their n_req_curve is infinite, not the "
      f"largest finite level)")

print("\nwhich level binds, publishing all nine:")
print(pd.Series(binding_level).value_counts().to_string())
pair = np.isfinite(curve_max) & np.isfinite(R_lvl[:, i50])
ratio = curve_max[pair] / np.maximum(R_lvl[pair, i50], 1)
print(f"\np50 binds for only {(binding_level == 'p50').sum()} / {len(binding_level)} countries.")
print(f"inflation from publishing all nine rather than p50 alone (over the {pair.sum()} countries "
      f"where both are finite):\n  median {np.median(ratio):.1f}x, P90 {np.quantile(ratio, .9):,.0f}x")

In [ ]:
# Four countries whose requirement peaks at different levels - the point of the section.
peak = {}
for j, p_ in enumerate(PERCENTILES):
    cand = np.where((binding_level == f"p{p_}") & (n_med > 1000) & np.isfinite(curve_max))[0]
    if len(cand):
        peak[p_] = cand[np.argmax(curve_max[cand])]
show = list(peak.items())[:5]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
ax = axes[0]
for p_, i in show:
    ax.plot(PERCENTILES, np.where(np.isinf(R_lvl[i]), 1e9, R_lvl[i]), marker="o", ms=4,
            label=f"{req.cc.iloc[i]} — peaks at p{p_}")
ax.set_yscale("log")
ax.set_xlabel("percentile level IQB is evaluated at")
ax.set_ylabel("measurements required")
ax.set_title("The hardest level is country-specific", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
for name, v, col in zip(SCENARIOS, SCENARIOS.values(), ["#2b6cb0", "#dd6b20", "#c53030"]):
    s = np.sort(np.nan_to_num(v, nan=1e9))
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), lw=2, color=col, label=name)
ax.set_xscale("log")
ax.set_xlim(10, 1e7)
ax.set_xlabel("measurements per period required")
ax.set_ylabel("share of countries covered")
ax.set_title("Publishing more levels raises the bar for everyone", fontsize=10)
ax.legend(fontsize=9, loc="lower right")
fig.tight_layout()
plt.show()

print("Because the requirement is computable per (country, level), an all-or-nothing gate throws")
print("information away: a country that cannot support a trustworthy p99 usually still has a")
print("solid p50. Publish the levels that clear the bar, band the ones that do not.")

## 10. How Far to Trust This

Three things can undermine the numbers above: countries where no sample size works, a
reconstruction step that may not be determinate, and the modelling assumptions.

### 10a. Countries No Sample Size Can Fix

If a country's pass rate for a heavy threshold sits *exactly* at the reporting level, the test is a
50/50 result at every $n$ and the requirement does not merely grow — it diverges. This is a property
of threshold scoring, not of data collection, and it is the one case where the honest answer is
"no number of measurements will help".

Two versions, with different answers. **Permanently** on a threshold means the country sits on a
threshold in its 24-month pooled distribution — a persistent problem. **Temporarily** on a threshold
means it happens to land there this month, which is far more common and is what produces the wide
confidence bands in notebook 02.

In [ ]:
t_month = true_scores(P_all, Q_HEADLINE)
req_month = required_n(1 - P_all, t_month, Q_HEADLINE, ratio_arr)
km = pd.DataFrame({"cc": cc_arr, "month": df.month.to_numpy(), "n_req": req_month})

print("per country-month requirement (p50, tol +/-0.05):")
print(f"  median {km.n_req.replace(np.inf, np.nan).median():,.0f}   "
      f"P90 {km.n_req.replace(np.inf, np.nan).quantile(.9):,.0f}   "
      f"P99 {km.n_req.replace(np.inf, np.nan).quantile(.99):,.0f}")
for thr in [1e4, 1e5, 1e6]:
    print(f"  share of country-months needing more than {thr:>11,.0f}: "
          f"{(km.n_req > thr).mean():6.1%}")

frac = km.groupby("cc").n_req.apply(lambda s: (s > 1e5).mean())
print(f"\ncountries temporarily on a threshold in at least one month: {(frac > 0).sum()} / {len(frac)}")
print(f"countries sitting on a threshold in a quarter or more of their months: {(frac >= 0.25).sum()}")

hard = req[req["n_req_0.05"] > 1e5].copy()
hard["have_enough"] = np.where(hard.n_month_median >= hard["n_req_0.05"], "yes", "no")
print(f"\ncountries permanently on a threshold ({len(hard)}), pooled over 24 months:")
print(hard.sort_values("n_req_0.05", ascending=False)
      [["cc", "country", "n_month_median", "iqb_p50", "margin", "binding_test",
        "n_req_0.05", "have_enough"]].to_string(index=False,
                                                float_format=lambda v: f"{v:,.5g}"))
print(f"\n{(hard.have_enough == 'yes').sum()} of them already collect enough to clear even that "
      f"requirement; {(hard.have_enough == 'no').sum()} cannot realistically get there.")

### 10b. The Weakest Link: Recovering the Pass Rate

Every number in this notebook rests on $\pi$, and $\pi$ is not reported — it is reconstructed by
interpolating between nine percentile knots (§4). That step deserves scrutiny, because the whole
apparatus is measuring distances of a few thousandths on the pass-rate scale.

Notebook 02's `make_cdf` collapses tied values and keeps the *top* probability. This notebook uses
the bracket that the definition of a quantile forces: for $t$ strictly between adjacent distinct
knot values $u_i < u_{i+1}$,

$$F(t) \;\in\; \bigl[\,q_{\max}(u_i),\; q_{\min}(u_{i+1})\,\bigr)$$

since $F(u_i) \ge q$ for every knot at $u_i$, and $F(t) < q$ for every knot at $u_{i+1}$.
Notebook 02's convention can leave that bracket just below a point mass — not a defect there, since
it is reconstructing a curve to re-invert, but it matters here because loss has a large point mass
at zero and the two loss thresholds are the binding test for roughly half of all countries.

Neither convention is right: nine knots genuinely do not determine $F$ between them. The question
is how much the answer moves, and the answer splits in two — **the aggregate is robust, the
per-country numbers often are not.**

In [ ]:
def pass_prob_variant(y_knots, t, tie="bracket"):
    y = np.asarray(y_knots, float)
    uy = np.unique(y)
    if tie == "bracket":
        return pass_prob(y_knots, t)
    up = np.array([(QLEV[y == v].max() if tie == "top" else QLEV[y == v].min()) for v in uy])
    F = (1.0 if t >= uy[0] else 0.0) if len(uy) == 1 else float(np.interp(t, uy, up))
    return float(np.clip(1.0 - F, 0.005, 0.995))


def variant_matrix(frame, tie="bracket", log_space=True):
    cols = []
    for m in METRICS:
        raw = frame[[f"{m}_p{p}" for p in PERCENTILES]].to_numpy(float)
        sign = 1 if m in ("download", "upload") else -1
        Y = to_quality(m, raw) if log_space else raw * sign
        for thr, _ in ATOMS[m]:
            t = float(to_quality(m, thr)) if log_space else float(thr * sign)
            cols.append(np.array([pass_prob_variant(Y[i], t, tie) for i in range(len(Y))]))
    return np.column_stack(cols)


def pool(P):
    rows = []
    for c in pd.unique(cc_arr):
        m = cc_arr == c
        w = n_arr[m]
        w = w / w.sum() if w.sum() > 0 else np.full(m.sum(), 1 / m.sum())
        rows.append((P[m] * w[:, None]).sum(axis=0))
    return np.array(rows)


VARIANTS = {"bracket, log space (this notebook)": dict(tie="bracket", log_space=True),
            "nb02 make_cdf (top prob at ties)":   dict(tie="top",     log_space=True),
            "lower envelope (bottom prob)":       dict(tie="bottom",  log_space=True),
            "bracket, linear value space":        dict(tie="bracket", log_space=False)}
variants = {}
for name, kw in VARIANTS.items():
    Pv = pool(variant_matrix(df, **kw))
    variants[name] = dict(P=Pv, req=required_n(1 - Pv, true_scores(Pv, Q_HEADLINE),
                                               Q_HEADLINE, ratio_c))

print("=== requirement under four reconstructions (p50, tol +/-0.05) ===")
print(f"{'reconstruction':36s}" + "".join(f"{k:>10s}" for k in ["P25", "P50", "P75", "P90"])
      + f"{'coverage@1k':>13s}")
for name, d in variants.items():
    s = pd.Series(d["req"]).replace(np.inf, np.nan)
    print(f"{name:36s}" + "".join(f"{s.quantile(x):>10,.0f}" for x in (.25, .5, .75, .9))
          + f"{(d['req'] <= 1000).mean():>13.1%}")

base = variants["bracket, log space (this notebook)"]
print("\nper-country verdict flips vs this notebook's choice:")
for name, d in variants.items():
    if d is base:
        continue
    print(f"  {name:36s} {((n_med >= base['req']) != (n_med >= d['req'])).sum():3d} / {len(n_med)}")

# A country's requirement is unresolved only if the conventions disagree about the test that
# SETS it, by more than that test's own margin. Comparing the largest disagreement across all ten
# tests against the binding test's margin counts countries whose disagreement sits on a test with
# margin to spare, and overstates the problem roughly threefold.
P_nb02_ = variants["nb02 make_cdf (top prob at ties)"]["P"]
spread_all = np.abs(P_nb02_ - P_c)
spread_bind = spread_all[np.arange(len(P_c)), kbind]
unresolved = spread_bind > req.margin.to_numpy()
straddle = (((P_c > 1 - Q_HEADLINE) != (P_nb02_ > 1 - Q_HEADLINE)) & HEAVY).any(axis=1)

print(f"\nmedian disagreement between conventions: {np.median(spread_all.max(axis=1)):.4f} "
      f"on the pass-rate scale")
print(f"{'countries where the binding test disagreement exceeds its own margin:':<62s}"
      f"{unresolved.sum():>4d} / {len(unresolved)}")
print(f"{'...of those, where the two conventions land on opposite sides:':<62s}"
      f"{straddle.sum():>4d} / {len(unresolved)}")
print(f"{'(unmatched: max disagreement over all 10 tests vs binding margin)':<62s}"
      f"{(spread_all.max(axis=1) > req.margin.to_numpy()).sum():>4d} / {len(unresolved)}  <- overcounts")
print("  -> for the first group, nine knots do not determine the margin, so their individual")
print("     requirement is unresolved rather than large.")

In [ ]:
# What that means for one country, and what notebook 02's extra inversion step costs.
P_nb02 = variants["nb02 make_cdf (top prob at ties)"]["P"]
req_nb02 = variants["nb02 make_cdf (top prob at ties)"]["req"]
req_base = req["n_req_0.05"].to_numpy()
ccs = req.cc.to_numpy()

flip = np.where((n_med >= req_base) != (n_med >= req_nb02))[0]
swing = flip[np.argsort(-np.abs(np.log10(np.maximum(req_base[flip], 1)) -
                               np.log10(np.maximum(req_nb02[flip], 1))))]
i = swing[0]
k = int(np.argmax(np.abs(P_c[i] - P_nb02[i])))
met, thr = ATOM_LABELS[k].split()[0], float(ATOM_LABELS[k].split()[-1])
sub = df[df.country_code == ccs[i]]
w = sub.sample_count.to_numpy(float)
knots = (sub[[f"{met}_p{p}" for p in PERCENTILES]].to_numpy(float) * (w / w.sum())[:, None]).sum(0)

print(f"=== {req.country.iloc[i]} ({ccs[i]}): binding test {ATOM_LABELS[k]}, "
      f"weight {ATOM_W[k]:.3f} ===")
print(f"pooled {met} percentiles as the data reports them:")
for j, (p_, v) in enumerate(zip(PERCENTILES, knots)):
    gap = j > 0 and min(knots[j - 1], v) < thr < max(knots[j - 1], v)
    print(f"   {met}_p{p_:<2d} = {v:.6f}" + ("   <-- the threshold falls in this gap" if gap else ""))
print(f"\n   pass rate, bracket convention : {P_c[i, k]:.4f} -> margin "
      f"{abs(P_c[i, k] - 0.5):.4f} -> needs {req_base[i]:,.0f}")
print(f"   pass rate, nb02 convention    : {P_nb02[i, k]:.4f} -> margin "
      f"{abs(P_nb02[i, k] - 0.5):.4f} -> needs {req_nb02[i]:,.0f}")
print(f"   it actually collects {n_med[i]:,.0f} per month")
print(f"\nBoth conventions agree that about half its measurements pass; they disagree on whether")
print(f"that is {P_c[i, k]:.2%} or {P_nb02[i, k]:.2%}. At the median that is the difference between a "
      f"50/50 result and a\nsettled answer. More measurement cannot resolve it - the ambiguity is in "
      f"the nine-knot\nsummary, not in the sample. One COUNTIF would.")

### 10c. The Modelling Assumptions

Two remaining choices, both of which turn out not to matter much: treating the four metrics as
independent, and the tolerance $\tau$. The dependence check recomputes the requirement under the
opposite extreme — all four quantiles perfectly comonotone, sharing a single CDF position.

In [ ]:
def required_n_comonotone(C, t, q, tol=TOL, conf=CONF):
    # Upper bound on what metric dependence can do: all ten tests share one CDF position, and all
    # are evaluated at the download-side n (upload has fewer, so this is the conservative side).
    order = np.argsort(C, axis=1)
    Cs = np.take_along_axis(C, order, axis=1)
    vals = np.concatenate([np.zeros((len(C), 1)), np.cumsum(ATOM_W[order], axis=1)], axis=1)
    ok = []
    for n in NGRID:
        r = int(np.clip(np.ceil(q * n), 1, n))
        sf = stats.beta.sf(Cs, r, n - r + 1)
        pr = np.empty_like(vals)
        pr[:, 0] = 1 - sf[:, 0]
        pr[:, -1] = sf[:, -1]
        pr[:, 1:-1] = sf[:, :-1] - sf[:, 1:]
        ok.append((np.clip(pr, 0, None) *
                   (np.abs(vals - t[:, None]) <= tol + 1e-12)).sum(axis=1) >= conf)
    ok = np.array(ok)
    allok = np.empty_like(ok)
    acc = np.ones(len(C), bool)
    for i in range(len(NGRID) - 1, -1, -1):
        acc = acc & ok[i]
        allok[i] = acc
    return np.where(allok.any(axis=0), NGRID[np.argmax(allok, axis=0)], np.inf).astype(float)


como = required_n_comonotone(C_c, t_c, Q_HEADLINE)
f = np.isfinite(como) & np.isfinite(req["n_req_0.05"])
print("metric dependence")
print(f"   median requirement, independent metrics : {np.median(req.loc[f, 'n_req_0.05']):>10,.0f}")
print(f"   median requirement, perfectly correlated: {np.median(como[f]):>10,.0f}")
print(f"   median ratio {np.median(como[f] / req.loc[f, 'n_req_0.05']):.2f}, "
      f"P90 ratio {np.quantile(como[f] / req.loc[f, 'n_req_0.05'], .9):.2f}"
      "  -> immaterial: one binding test sets the requirement, and dependence has nothing to act on.")
print("\ntolerance")
for x in TOLS:
    print(f"   tau = {x:<5} -> median requirement "
          f"{req[f'n_req_{x}'].replace(np.inf, np.nan).median():>8,.0f}")

## 11. Does the Model Match Reality?

Everything so far is a model of sampling error. The 24-month panel gives an independent read: if
the model is right, a country's month-to-month movement should be *at least* as large as the
predicted sampling noise, with the excess being real change in the network or in who is measuring.

**MAD is the right statistic for the observed series**, for the reason given in §7 — a country's
IQB history is mostly a flat line on the lattice with occasional one-test jumps, so a standard
deviation reports the jumps while MAD reports the typical state. Both are shown.

The column that matters is the last one: the share of observed month-to-month variance that
sampling error can account for.

In [ ]:
K_MAD = 1.4826    # scales MAD to a standard-deviation equivalent under normality

hist = pd.DataFrame({"cc": cc_arr, "iqb": t_month})
g = hist.groupby("cc")
emp = pd.DataFrame({
    "mad_obs": g.iqb.apply(lambda s: K_MAD * np.median(np.abs(s - np.median(s)))),
    "sd_obs": g.iqb.std(),
    "n_months": g.size(),
}).reset_index().merge(req[["cc", "n_month_median", "sd_at_current_n"]], on="cc")
emp = emp[emp.n_months >= 12]
emp["bin"] = pd.cut(np.log10(emp.n_month_median.clip(lower=1)), [0, 1, 2, 3, 4, 5, 6, 8],
                    labels=["<10", "10-100", "100-1k", "1k-10k", "10k-100k", "100k-1M", ">1M"])

# The variance share belongs to a country, so aggregating it means taking the median of the
# per-country ratios. Dividing a bin's mean predicted SD by its median observed SD is not a share
# of anything, and let one or two countries sitting on a threshold make sd_predicted rise with n.
emp["sampling_share"] = (emp.sd_at_current_n ** 2
                         / emp.sd_obs.replace(0.0, np.nan) ** 2).clip(upper=1)
tab = emp.groupby("bin", observed=True).agg(
    countries=("cc", "size"), median_n=("n_month_median", "median"),
    mad_observed=("mad_obs", "median"), sd_observed=("sd_obs", "median"),
    sd_predicted=("sd_at_current_n", "median"),
    sampling_share_of_var=("sampling_share", "median"))
print("observed month-to-month spread vs predicted sampling noise")
print(tab.to_string(float_format=lambda v: f"{v:,.4g}"))
flat = int((emp.sd_obs == 0).sum())
over = int((emp.sd_at_current_n > emp.sd_obs).sum())
print(f"\n{flat} of {len(emp)} countries never moved at all across their months (share undefined, "
      f"excluded above).")
print(f"predicted sampling SD exceeds observed SD for {over} countries - "
      f"{over - int(((emp.sd_at_current_n > emp.sd_obs) & (emp.sd_obs == 0)).sum())} of them "
      f"substantively, all at small n,\nwhere the 24-month pooled distribution is a wider mixture "
      f"than any single month and over-predicts the spread.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

ax = axes[0]
ax.scatter(emp.n_month_median, emp.sd_obs, s=15, alpha=0.5, color="#2b6cb0",
           label="observed SD across months")
ax.scatter(emp.n_month_median, emp.sd_at_current_n, s=15, alpha=0.5, color="#c53030",
           label="predicted sampling SD")
ax.set_xscale("log")
ax.set_xlabel("median monthly measurements")
ax.set_ylabel("SD of the p50 IQB score")
ax.set_title("Predicted sampling noise vanishes; observed volatility does not", fontsize=10)
ax.legend(fontsize=8)

ax = axes[1]
x = np.arange(len(tab))
ax.bar(x, tab.sampling_share_of_var, color="#dd6b20", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(tab.index, rotation=20, fontsize=8)
ax.set_ylabel("share of observed variance explained by sampling")
ax.set_xlabel("median monthly measurements")
ax.set_title("Past a few thousand measurements, month-to-month movement\nis no longer a sampling "
             "artefact", fontsize=10)
for xi, v in zip(x, tab.sampling_share_of_var):
    ax.text(xi, v + 0.01, f"{v:.0%}", ha="center", fontsize=8)
fig.tight_layout()
plt.show()

below = tab.index[np.argmax(tab.sampling_share_of_var.to_numpy() < 0.05)]
print(f"Sampling error falls below 5% of observed variance in the '{below}' bin.")
print("Beyond that, more measurements do not make a country's reported score steadier. The")
print("residual movement is real change in the network or in the measuring population - a signal")
print("to interpret, not noise to average away. Predicted noise is smaller than observed for")
print(f"{len(emp) - over} of {len(emp)} countries, so the model is not over-claiming except at "
      f"the smallest n.")

## 12. What To Do

**The answer.** To be within ±0.05 of the true score 95% of the time, the median country needs
about **560** download-side measurements publishing p50 alone, **2,400** publishing p25/p50/p75,
and **5,400** publishing all nine levels. At the 90th percentile of countries those become roughly
17,000, 86,000 and 114,000. On today's monthly volumes that is 185, 138 and 111 of 245 countries.
Against a single month rather than the pooled distribution, add roughly a quarter at the median
and a factor of two in the tail (§10a).

**Five things follow, in order of how much they help.**

1. **Gate per (country, percentile level), not on a global n.** Every requirement is computable in
   closed form from data already on hand, so a fixed cutoff is strictly worse: it suppresses
   well-separated small countries and passes large ones that sit on a threshold. Gating per country is better;
   per level better still, since a country that cannot support a trustworthy p99 usually still has
   a solid p50. The exported CSV carries a requirement for each level and a `levels_supported`
   column.
2. **Emit the ten pass rates from the aggregation pipeline.** This is the cheapest fix here and it
   removes an error no sample size can. Because IQB at level $q$ is
   $\sum_k w_k \mathbf{1}\{q > 1-\pi_k\}$, the ten pass rates are a **sufficient statistic for the
   entire IQB-versus-percentile curve** — they determine the score at every level exactly. They are
   one `COUNTIF`/`COUNT` per threshold in the existing aggregation. Today $\pi$ must be
   interpolated between nine knots, and §10b shows that for 22 of 245 countries that interpolation
   is less certain than the margin it is measuring. Ten extra columns would make this analysis
   exact, would fix the tails where 12% of thresholds fall outside the reported range entirely,
   and would make the sampling model checkable for the first time (§6c).
3. **Correct or flag the small-sample bias — do not just widen the error bar.** A tolerance band
   describes spread around the wrong centre. 24 countries are currently published below their true
   score by more than the tolerance itself. Two options: publish the bias-corrected estimate
   (the sampling distribution is in closed form, so $\widehat{\mathrm{IQB}} - \text{bias}$ costs
   nothing), or suppress the point estimate and publish the band. What is not defensible is
   ranking a country with 15 measurements against one with 15 million as though the two numbers
   mean the same thing. The CSV carries `bias_at_current_n`.
4. **Publish a band, not a point, for countries sitting on a threshold.** Their score is indeterminate at any
   sample size. Notebook 02's CI machinery already produces the band.
5. **Do not collect measurements beyond ~10³/month for stability's sake.** Above that, sampling error
   accounts for well under 1% of the month-to-month movement actually observed; the rest is real
   change — mostly within networks rather than shifts in which networks got measured.

**A structural note, outside the question but visible from it.** All of the pain here comes from
IQB being a sum of hard indicator functions. A score that ramped smoothly across each threshold
instead of stepping would make the estimate continuous in the data, remove the divergent
requirement entirely, remove the bias in §8b, and shrink the typical requirement by roughly the
ratio of the step to the ramp width. That is a scoring-design decision rather than a measurement
one, but the sample-size problem is largely created by how the score is designed.

### Caveats

- **These are pooled-distribution requirements, not single-month ones.** The margin every number
  rests on is estimated from 24 months pooled, while the score being published covers one month.
  A country's own pass rate moves month to month by about as much as the margin a few hundred
  measurements resolve, so §10a's monthly figures (median 706, P90 29,460) are the operational
  ones and the headline is a minimum.
- **Effective sample size is smaller than the raw count.** The spread $\sqrt{n\pi(1-\pi)}$ assumes
  independent measurements. M-Lab tests are self-selected and clustered: the median country's
  measurements sit in about 4 effective ASNs, with its largest ASN supplying roughly a third of
  them, and pass rates differ across a country's ASNs with a spread near 0.09. Every figure here
  is therefore a **lower bound** on what is really needed.
- **The aggregate numbers are firmer than the per-country ones.** The median and quartile
  requirements are stable across four reconstruction conventions, but for 22 of 245 countries the
  disagreement on the test that sets their requirement exceeds that test's own margin. Treat those
  as unresolved rather than large; the fix is recommendation 2, not more measurements.
- Thresholds outside the p1–p99 range yield a bound, recorded as a margin of 0.495, which
  understates how safe those tests are. The all-nine-levels figure is the one most exposed to the
  convention used at the edges: across plausible choices of that bound its median moves between
  about 5,100 and 7,200 and its 90th percentile between 114,000 and 201,000. The p50 figure does
  not move at all.
- The rank convention is $r = \lceil qn \rceil$; others differ only at very small $n$.


In [ ]:
export = req.copy()
export = export.rename(columns={"n_month_median": "n_down_month_median"})
# Requirements are quoted in download-side measurements; the upload side needs the country's own
# ratio applied, since that is a different set of tests with a different count.
export["n_req_upload_side"] = np.ceil(export["n_req_0.05"] * export.up_ratio).replace(np.inf, np.nan)
export["months_to_pool"] = np.ceil(export["n_req_0.05"] /
                                   export.n_down_month_median.clip(lower=1)).replace(np.inf, np.nan)
export["verdict_p50"] = np.select(
    [export.n_down_month_median >= export["n_req_0.05"],
     export.n_down_month_median * 12 >= export["n_req_0.05"],
     export["n_req_0.05"] > 1e5],
    ["monthly data is sufficient",
     "sufficient if months are pooled",
     "sits on a threshold: no sample size is enough, publish a range"],
    default="short of data: pool or suppress")

for j, p_ in enumerate(PERCENTILES):
    export[f"n_req_p{p_}"] = R_lvl[:, j]
export["n_req_curve"] = curve_max
export["binding_level"] = binding_level
export["levels_supported"] = [
    " ".join(f"p{p_}" for j, p_ in enumerate(PERCENTILES) if nm >= R_lvl[i, j]) or "none"
    for i, nm in enumerate(n_med)]
export["margin_unresolved"] = unresolved      # see §10b (binding test, matched comparison)
export["bias_at_current_n"] = bias_now        # see §8b - sign matters, not just size
export["curve_reachable"] = np.isfinite(curve_max)

cols = (["cc", "country", "n_down_month_median", "up_ratio", "iqb_p50", "margin", "binding_test",
         "n_req_0.02", "n_req_0.05", "n_req_0.1", "n_req_upload_side", "months_to_pool",
         "sd_at_current_n", "mad_at_current_n", "p_ok_at_current_n", "bias_at_current_n",
         "verdict_p50"]
        + [f"n_req_p{p_}" for p_ in PERCENTILES]
        + ["n_req_curve", "curve_reachable", "binding_level", "levels_supported",
           "margin_unresolved"])
out_path = OUT_DIR / "03_sample_size_requirements.csv"
export[cols].sort_values("n_req_0.05", ascending=False).to_csv(out_path, index=False)

print(f"wrote {out_path}  ({len(export)} countries, {len(cols)} columns)\n")
print(export.verdict_p50.value_counts().to_string())
n_ok = np.array([0 if v == "none" else len(v.split()) for v in export.levels_supported])
print(f"\nlevels each country can support today (of 9): median {np.median(n_ok):.0f}")
print(f"  all nine: {(n_ok == 9).sum():3d}   five or more: {(n_ok >= 5).sum():3d}   "
      f"none: {(n_ok == 0).sum():3d}")
print(f"  margin unresolved (see 10b): {export.margin_unresolved.sum()}   "
      f"bias above tolerance (see 8b): {(export.bias_at_current_n.abs() > TOL).sum()}")
print("\nexample rows:")
print(export.sort_values("n_down_month_median", ascending=False)
      [["cc", "n_down_month_median", "iqb_p50", "margin", "n_req_p50", "n_req_curve",
        "binding_level", "levels_supported"]].head(8).to_string(index=False,
                                                                float_format=lambda v: f"{v:,.4g}"))